In [30]:
import requests
import pandas as pd

In [31]:
def get_coordinates(city_name):

    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": city_name,
        "count": 10,
        "language": "en",
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    if not data.get("results"):
        print(f"{city_name} not found!")
        return None

    for city in data["results"]:

        if city.get("country_code") == "PK":

            return {
                "city": city["name"],
                "country": city.get("country"),
                "latitude": city["latitude"],
                "longitude": city["longitude"]
            }

    print(f"{city_name} not found in Pakistan!")
    return None

In [32]:
cities = [
    "Islamabad",
    "Lahore",
    "Karachi",
    "Peshawar",
    "Quetta"
]

city_coordinates = []

for city in cities:

    result = get_coordinates(city)

    if result is not None:
        city_coordinates.append(result)

coordinates_df = pd.DataFrame(city_coordinates)

coordinates_df

,city,country,latitude,longitude
0,Islamabad,Pakistan,33.72148,73.04329
1,Lahore,Pakistan,31.55800,74.35071
2,Karachi,Pakistan,24.86080,67.01040
3,Peshawar,Pakistan,34.00800,71.57849
4,Quetta,Pakistan,30.18414,67.00141


In [33]:
hourly_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "weather_code",
    "wind_speed_10m",
    "wind_direction_10m",
    "cloud_cover",
    "pressure_msl"
]

In [34]:
def extract_weather_data(city_name, year):

    city_data = coordinates_df[
        coordinates_df["city"] == city_name
    ].iloc[0]

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": city_data["latitude"],
        "longitude": city_data["longitude"],
        "start_date": f"{year}-01-01",
        "end_date": f"{year}-12-31",
        "hourly": hourly_variables,
        "timezone": "Asia/Karachi"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    data = response.json()

    weather_data = pd.DataFrame(data["hourly"])

    weather_data["city"] = city_name
    weather_data["year"] = year

    return weather_data

In [36]:
years = range(2021, 2026)

all_weather_data = []

for city in cities:

    for year in years:

        print(f"Extracting {city} - {year}")

        weather_data = extract_weather_data(city, year)

        all_weather_data.append(weather_data)

final_df = pd.concat(all_weather_data, ignore_index=True)

Extracting Islamabad - 2021
Extracting Islamabad - 2022
Extracting Islamabad - 2023
Extracting Islamabad - 2024
Extracting Islamabad - 2025
Extracting Lahore - 2021
Extracting Lahore - 2022
Extracting Lahore - 2023
Extracting Lahore - 2024
Extracting Lahore - 2025
Extracting Karachi - 2021
Extracting Karachi - 2022
Extracting Karachi - 2023
Extracting Karachi - 2024
Extracting Karachi - 2025
Extracting Peshawar - 2021
Extracting Peshawar - 2022
Extracting Peshawar - 2023
Extracting Peshawar - 2024
Extracting Peshawar - 2025
Extracting Quetta - 2021
Extracting Quetta - 2022
Extracting Quetta - 2023
Extracting Quetta - 2024
Extracting Quetta - 2025


In [37]:
print("Dataset Shape:", final_df.shape)

final_df.head()

Dataset Shape: (219120, 13)


,time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,wind_speed_10m,wind_direction_10m,cloud_cover,pressure_msl,city,year
0,2021-01-01T00:00,9.2,36,5.0,0.0,0.0,0,9.5,323,3,1016.7,Islamabad,2021
1,2021-01-01T01:00,9.3,27,4.7,0.0,0.0,0,10.0,319,0,1016.5,Islamabad,2021
2,2021-01-01T02:00,9.6,20,4.6,0.0,0.0,0,10.5,322,0,1016.1,Islamabad,2021
3,2021-01-01T03:00,9.6,18,4.6,0.0,0.0,1,10.0,322,35,1015.6,Islamabad,2021
4,2021-01-01T04:00,9.3,17,4.4,0.0,0.0,1,9.2,321,39,1015.4,Islamabad,2021


In [38]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219120 entries, 0 to 219119
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   time                  219120 non-null  object 
 1   temperature_2m        219120 non-null  float64
 2   relative_humidity_2m  219120 non-null  int64  
 3   apparent_temperature  219120 non-null  float64
 4   precipitation         219120 non-null  float64
 5   rain                  219120 non-null  float64
 6   weather_code          219120 non-null  int64  
 7   wind_speed_10m        219120 non-null  float64
 8   wind_direction_10m    219120 non-null  int64  
 9   cloud_cover           219120 non-null  int64  
 10  pressure_msl          219120 non-null  float64
 11  city                  219120 non-null  object 
 12  year                  219120 non-null  int64  
dtypes: float64(6), int64(5), object(2)
memory usage: 21.7+ MB


In [39]:
# Check missing values
print("Missing Values:")
print(final_df.isnull().sum())

# Check complete duplicate records
print("\nDuplicate Records:", final_df.duplicated().sum())

# Check duplicate city timestamps
print("\nDuplicate City-Time Records:",
      final_df.duplicated(subset=["city", "time"]).sum())

Missing Values:
time                    0
temperature_2m          0
relative_humidity_2m    0
apparent_temperature    0
precipitation           0
rain                    0
weather_code            0
wind_speed_10m          0
wind_direction_10m      0
cloud_cover             0
pressure_msl            0
city                    0
year                    0
dtype: int64

Duplicate Records: 0

Duplicate City-Time Records: 0


In [40]:
print("Records per City:")
print(final_df["city"].value_counts())

print("\nRecords per Year:")
print(final_df["year"].value_counts().sort_index())

print("\nRecords per City and Year:")
print(final_df.groupby(["city", "year"]).size().unstack())

Records per City:
city
Islamabad    43824
Lahore       43824
Karachi      43824
Peshawar     43824
Quetta       43824
Name: count, dtype: int64

Records per Year:
year
2021    43800
2022    43800
2023    43800
2024    43920
2025    43800
Name: count, dtype: int64

Records per City and Year:
year       2021  2022  2023  2024  2025
city                                   
Islamabad  8760  8760  8760  8784  8760
Karachi    8760  8760  8760  8784  8760
Lahore     8760  8760  8760  8784  8760
Peshawar   8760  8760  8760  8784  8760
Quetta     8760  8760  8760  8784  8760


In [41]:
final_df["time"] = pd.to_datetime(final_df["time"])

final_df = final_df.sort_values(
    by=["city", "time"]
).reset_index(drop=True)

In [42]:
# Extract date and time features

final_df["month"] = final_df["time"].dt.month
final_df["day"] = final_df["time"].dt.day
final_df["hour"] = final_df["time"].dt.hour
final_df["day_of_week"] = final_df["time"].dt.day_name()


# Create seasonal feature

season_mapping = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
}

final_df["season"] = final_df["month"].map(season_mapping)

In [43]:
print("Final Dataset Shape:", final_df.shape)

print("\nDataset Information:")
final_df.info()

print("\nSeason Distribution:")
print(final_df["season"].value_counts())

print("\nMissing Values:")
print(final_df.isnull().sum())

Final Dataset Shape: (219120, 18)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219120 entries, 0 to 219119
Data columns (total 18 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   time                  219120 non-null  datetime64[ns]
 1   temperature_2m        219120 non-null  float64       
 2   relative_humidity_2m  219120 non-null  int64         
 3   apparent_temperature  219120 non-null  float64       
 4   precipitation         219120 non-null  float64       
 5   rain                  219120 non-null  float64       
 6   weather_code          219120 non-null  int64         
 7   wind_speed_10m        219120 non-null  float64       
 8   wind_direction_10m    219120 non-null  int64         
 9   cloud_cover           219120 non-null  int64         
 10  pressure_msl          219120 non-null  float64       
 11  city                  219120 non-null  object        
 12  ye

In [44]:
!git status


On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Pakistan_Weather_Data_Analytics.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
